## Anna Giczewska ST544 - Project 2
Date 3/27/2026  

This project has two main parts. In Part I, I create and test a custom PySpark class called SparkDataCheck that helps check and summarize data quality in Spark SQL style DataFrames. In Part II, I analyze NFL quarterback data using both pandas-on-Spark and Spark SQL DataFrames.

Throughout the notebook, I include code, output, and short explanations to describe what I am doing and why each step is included.

## Part I - Creating and Testing the SparkDataCheck Class

In this part of the project, I work with a Spark SQL style DataFrame and create a custom Python class that wraps the DataFrame and provides methods for checking data quality and summarizing the data.

The class is stored in a separate .py file, and I import it into this notebook so I can test each method here.

In [6]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#start Spark Session for this project
spark = SparkSession.builder.appName("Project2").getOrCreate()
#turn off ANSI mode to avoid certainstrict Spark errors
spark.conf.set("spark.sql.ansi.enabled", "false")

In [2]:
#import importlib so I can reload my Python file after edditing it
import importlib
#import my claddfile
import spark_data_check
#reload the file so the notebook uses the newest version
importlib.reload(spark_data_check)

#Import the SparkDataCheck class from my file
from spark_data_check import SparkDataCheck

# Show which file is being imported
print(spark_data_check.__file__)

# Check which methods exist on the class itself
print(hasattr(SparkDataCheck, "from_csv"))
print(hasattr(SparkDataCheck, "from_pandas"))
print(hasattr(SparkDataCheck, "check_numeric_range"))
print(hasattr(SparkDataCheck, "check_string_levels"))
print(hasattr(SparkDataCheck, "check_missing"))
print(hasattr(SparkDataCheck, "summarize_numeric_min_max"))
print(hasattr(SparkDataCheck, "count_string_levels"))

/home/jupyter-agiczew@ncsu.edu/st554_project2/spark_data_check.py
True
True
True
True
True
True
True


In [3]:
air_url = "https://www4.stat.ncsu.edu/online/datasets/air.csv"
#read the CSV from the website into a pandas DataFrame
air_download = pd.read_csv(air_url)
#save a local copy of air.csv file in my project folder
air_download.to_csv("air.csv", index=False)

# Check the first few rows and column names
print(air_download.columns.tolist())
air_download.head()

['Unnamed: 0', 'Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [4]:
#createa SparkData Object by reading CSV with Spark
air_check = SparkDataCheck.from_csv(spark, "air.csv")
#confirm object creation
type(air_check)

spark_data_check.SparkDataCheck

In [5]:
#show data
air_check.df.show(5)
air_check.df.printSchema()

+----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|2026-03-27 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|2026-03-27 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|2026-03-27 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         

In [6]:
#read CSV into pandas DataFrame
air_pd = pd.read_csv("air.csv")
#Create a SparkDataCheck object from the pandas DataFrame
air_check_pd = SparkDataCheck.from_pandas(spark, air_pd)
#confirm object was created
type(air_check_pd)


spark_data_check.SparkDataCheck

In [7]:
air_check_pd.df.show(5)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

In [8]:
# Create helper columns for testing the validation methods
air_check.df = (
    air_check.df
    .withColumn(
        "time_period",
        F.when(F.col("Time") < "12:00:00", "morning")
         .when(F.col("Time") < "18:00:00", "afternoon")
         .otherwise("evening")
    )
    .withColumn(
        "temp_group",
        F.when(F.col("T").isNull(), F.lit(None))
         .when(F.col("T") < 10, "cold")
         .when(F.col("T") < 20, "mild")
         .otherwise("warm")
    )
    .withColumn(
        "co_clean",
        F.when(F.col("CO(GT)") == -200, F.lit(None))
         .otherwise(F.col("CO(GT)"))
    )
)

# Show the new helper columns
air_check.df.select("Time", "T", "CO(GT)", "time_period", "temp_group", "co_clean").show(10)

+-------------------+----+------+-----------+----------+--------+
|               Time|   T|CO(GT)|time_period|temp_group|co_clean|
+-------------------+----+------+-----------+----------+--------+
|2026-03-27 18:00:00|13.6|   2.6|    evening|      mild|     2.6|
|2026-03-27 19:00:00|13.3|   2.0|    evening|      mild|     2.0|
|2026-03-27 20:00:00|11.9|   2.2|    evening|      mild|     2.2|
|2026-03-27 21:00:00|11.0|   2.2|    evening|      mild|     2.2|
|2026-03-27 22:00:00|11.2|   1.6|    evening|      mild|     1.6|
|2026-03-27 23:00:00|11.2|   1.2|    evening|      mild|     1.2|
|2026-03-27 00:00:00|11.3|   1.2|    morning|      mild|     1.2|
|2026-03-27 01:00:00|10.7|   1.0|    morning|      mild|     1.0|
|2026-03-27 02:00:00|10.7|   0.9|    morning|      mild|     0.9|
|2026-03-27 03:00:00|10.3|   0.6|    morning|      mild|     0.6|
+-------------------+----+------+-----------+----------+--------+
only showing top 10 rows


**Method 1: Check check_numeric_range()**

In [24]:
# Check whether temperature is in a reasonable range
air_check.check_numeric_range("T", lower=0, upper=40, new_col_name="temp_ok")
air_check.df.select("T", "temp_ok").show(10)

+----+-------+
|   T|temp_ok|
+----+-------+
|13.6|   true|
|13.3|   true|
|11.9|   true|
|11.0|   true|
|11.2|   true|
|11.2|   true|
|11.3|   true|
|10.7|   true|
|10.7|   true|
|10.3|   true|
+----+-------+
only showing top 10 rows


In [25]:
# Check whether relative humidity is at most 100
air_check.check_numeric_range("RH", upper=100, new_col_name="rh_ok")
air_check.df.select("RH", "rh_ok").show(10)

+----+-----+
|  RH|rh_ok|
+----+-----+
|48.9| true|
|47.7| true|
|54.0| true|
|60.0| true|
|59.6| true|
|59.2| true|
|56.8| true|
|60.0| true|
|59.7| true|
|60.2| true|
+----+-----+
only showing top 10 rows


In [26]:
# Test a bad input using a string column
air_check.check_numeric_range("time_period", lower=1, upper=10)

Column 'time_period' is not numeric.


In [27]:
# Check whether AH is at least 0.5
air_check.check_numeric_range("AH", lower=0.5, new_col_name="AH_ge_0_5")
air_check.df.select("AH", "AH_ge_0_5").show(10)

+------+---------+
|    AH|AH_ge_0_5|
+------+---------+
|0.7578|     true|
|0.7255|     true|
|0.7502|     true|
|0.7867|     true|
|0.7888|     true|
|0.7848|     true|
|0.7603|     true|
|0.7702|     true|
|0.7648|     true|
|0.7517|     true|
+------+---------+
only showing top 10 rows


In [28]:
# Check whether co_clean is between 0 and 5
air_check.check_numeric_range("co_clean", lower=0, upper=5, new_col_name="co_clean_0_5")
air_check.df.select("co_clean", "co_clean_0_5").show(10)

+--------+------------+
|co_clean|co_clean_0_5|
+--------+------------+
|     2.6|        true|
|     2.0|        true|
|     2.2|        true|
|     2.2|        true|
|     1.6|        true|
|     1.2|        true|
|     1.2|        true|
|     1.0|        true|
|     0.9|        true|
|     0.6|        true|
+--------+------------+
only showing top 10 rows


**Method 2: Check check_string_levels()**

In [29]:
# Check whether time_period belongs to the expected levels
air_check.check_string_levels(
    "time_period",
    ["morning", "afternoon", "evening"],
    new_col_name="time_period_ok"
)
air_check.df.select("time_period", "time_period_ok").show(10)

+-----------+--------------+
|time_period|time_period_ok|
+-----------+--------------+
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    evening|          true|
|    morning|          true|
|    morning|          true|
|    morning|          true|
|    morning|          true|
+-----------+--------------+
only showing top 10 rows


In [30]:
# Check whether temp_group belongs to the expected levels
air_check.check_string_levels(
    "temp_group",
    ["cold", "mild", "warm"],
    new_col_name="temp_group_ok"
)
air_check.df.select("temp_group", "temp_group_ok").show(10)

+----------+-------------+
|temp_group|temp_group_ok|
+----------+-------------+
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
|      mild|         true|
+----------+-------------+
only showing top 10 rows


In [31]:
# Test a bad input using a numeric column
air_check.check_string_levels("T", ["cold", "warm"])

Column 'T' is not a string column.


In [32]:
# Check stricter valid levels for time_period
air_check.check_string_levels(
    "time_period",
    ["morning", "afternoon"],
    new_col_name="time_period_day_only"
)
air_check.df.select("time_period", "time_period_day_only").show(10)

+-----------+--------------------+
|time_period|time_period_day_only|
+-----------+--------------------+
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    evening|               false|
|    morning|                true|
|    morning|                true|
|    morning|                true|
|    morning|                true|
+-----------+--------------------+
only showing top 10 rows


In [33]:
# Check stricter valid levels for temp_group
air_check.check_string_levels(
    "temp_group",
    ["cold", "warm"],
    new_col_name="temp_group_cold_warm"
)
air_check.df.select("temp_group", "temp_group_cold_warm").show(10)

+----------+--------------------+
|temp_group|temp_group_cold_warm|
+----------+--------------------+
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
|      mild|               false|
+----------+--------------------+
only showing top 10 rows


**Method 3: check_missing()**

In [34]:
# Check missing values in the cleaned CO column
air_check.check_missing("co_clean", new_col_name="co_missing")
air_check.df.select("CO(GT)", "co_clean", "co_missing").show(15)

+------+--------+----------+
|CO(GT)|co_clean|co_missing|
+------+--------+----------+
|   2.6|     2.6|     false|
|   2.0|     2.0|     false|
|   2.2|     2.2|     false|
|   2.2|     2.2|     false|
|   1.6|     1.6|     false|
|   1.2|     1.2|     false|
|   1.2|     1.2|     false|
|   1.0|     1.0|     false|
|   0.9|     0.9|     false|
|   0.6|     0.6|     false|
|-200.0|    NULL|      true|
|   0.7|     0.7|     false|
|   0.7|     0.7|     false|
|   1.1|     1.1|     false|
|   2.0|     2.0|     false|
+------+--------+----------+
only showing top 15 rows


In [35]:
# Create another column with some artificial missing values for testing
air_check.df = air_check.df.withColumn(
    "ah_test",
    F.when(F.col("AH") < 0.5, F.lit(None)).otherwise(F.col("AH"))
)

# Check missing values in the new test column
air_check.check_missing("ah_test", new_col_name="ah_test_missing")
air_check.df.select("AH", "ah_test", "ah_test_missing").show(15)

+------+-------+---------------+
|    AH|ah_test|ah_test_missing|
+------+-------+---------------+
|0.7578| 0.7578|          false|
|0.7255| 0.7255|          false|
|0.7502| 0.7502|          false|
|0.7867| 0.7867|          false|
|0.7888| 0.7888|          false|
|0.7848| 0.7848|          false|
|0.7603| 0.7603|          false|
|0.7702| 0.7702|          false|
|0.7648| 0.7648|          false|
|0.7517| 0.7517|          false|
|0.7465| 0.7465|          false|
|0.7366| 0.7366|          false|
|0.7353| 0.7353|          false|
|0.7417| 0.7417|          false|
|0.7408| 0.7408|          false|
+------+-------+---------------+
only showing top 15 rows


In [36]:
# Check missingness in T
air_check.check_missing("T", new_col_name="T_missing")
air_check.df.select("T", "T_missing").show(10)

+----+---------+
|   T|T_missing|
+----+---------+
|13.6|    false|
|13.3|    false|
|11.9|    false|
|11.0|    false|
|11.2|    false|
|11.2|    false|
|11.3|    false|
|10.7|    false|
|10.7|    false|
|10.3|    false|
+----+---------+
only showing top 10 rows


In [37]:
# Check missingness in time_period
air_check.check_missing("time_period", new_col_name="time_period_missing")
air_check.df.select("time_period", "time_period_missing").show(10)

+-----------+-------------------+
|time_period|time_period_missing|
+-----------+-------------------+
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    evening|              false|
|    morning|              false|
|    morning|              false|
|    morning|              false|
|    morning|              false|
+-----------+-------------------+
only showing top 10 rows


In [38]:
# Check missingness in temp_group
air_check.check_missing("temp_group", new_col_name="temp_group_missing")
air_check.df.select("temp_group", "temp_group_missing").show(10)

+----------+------------------+
|temp_group|temp_group_missing|
+----------+------------------+
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
|      mild|             false|
+----------+------------------+
only showing top 10 rows


**Method 4: Check summarize_numeric_min_max()**

In [12]:
# Summarize one numeric column without grouping
result1 = air_check.summarize_numeric_min_max("T")

# Check the output type
print(type(result1))

# Show the result
result1

<class 'pandas.core.frame.DataFrame'>


,T_min,T_max
0,-200.0,44.6


In [13]:
# Summarize one numeric column grouped by a string column
result2 = air_check.summarize_numeric_min_max("RH", group_by="time_period")

# Check the output type
print(type(result2))

# Show the result
result2

<class 'pandas.core.frame.DataFrame'>


,time_period,RH_min,RH_max
0,afternoon,-200.0,88.7
1,evening,-200.0,87.2
2,morning,-200.0,87.1


In [14]:
# Summarize all numeric columns without grouping
result3 = air_check.summarize_numeric_min_max()

# Check the output type
print(type(result3))

# Show the result
result3

26/03/27 22:11:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


<class 'pandas.core.frame.DataFrame'>


,Unnamed: 0_min,Unnamed: 0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,T_min,T_max,RH_min,RH_max,AH_min,AH_max,co_clean_min,co_clean_max,ah_test_min,ah_test_max
0,0,9356,-200.0,11.9,-200,2040,-200,1189,-200.0,63.7,...,-200.0,44.6,-200.0,88.7,-200.0,2.231,0.1,11.9,0.5002,2.231


In [15]:
# Summarize all numeric columns grouped by a string column
result4 = air_check.summarize_numeric_min_max(group_by="time_period")

# Check the output type
print(type(result4))

# Show the result
result4

<class 'pandas.core.frame.DataFrame'>


,time_period,Unnamed: 0_min,Unnamed: 0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,...,T_min,T_max,RH_min,RH_max,AH_min,AH_max,co_clean_min,co_clean_max,ah_test_min,ah_test_max
0,afternoon,18,9356,-200.0,8.4,-200,1915,-200,832,-200.0,...,-200.0,44.6,-200.0,88.7,-200.0,2.2310,0.1,8.4,0.5002,2.2310
1,evening,0,9341,-200.0,11.9,-200,2040,-200,1189,-200.0,...,-200.0,40.6,-200.0,87.2,-200.0,2.1107,0.1,11.9,0.5007,2.1107
2,morning,6,9353,-200.0,8.1,-200,1961,-200,1129,-200.0,...,-200.0,38.3,-200.0,87.1,-200.0,2.1719,0.1,8.1,0.5007,2.1719


In [16]:
# Test a bad input using a string column
air_check.summarize_numeric_min_max("time_period")

Column 'time_period' is not numeric.


**Method 5: Check count_string_levels()**

In [17]:
# Count levels of one string column
result5 = air_check.count_string_levels("time_period")

# Check the output type
print(type(result5))

# Show the result
result5

<class 'pandas.core.frame.DataFrame'>


,time_period,count
0,afternoon,2337
1,evening,2340
2,morning,4680


In [18]:
# Count levels of another string column
result6 = air_check.count_string_levels("temp_group")

#show the results
result6

,temp_group,count
0,cold,2032
1,mild,3577
2,warm,3748


In [19]:
# Count combinations of two string columns
result7 = air_check.count_string_levels("time_period", "temp_group")

# Check the output type
print(type(result7))

# Show the result
result7

<class 'pandas.core.frame.DataFrame'>


,time_period,temp_group,count
0,afternoon,cold,281
1,afternoon,mild,761
2,afternoon,warm,1295
3,evening,cold,484
4,evening,mild,884
5,evening,warm,972
6,morning,cold,1267
7,morning,mild,1932
8,morning,warm,1481


In [20]:
# Test a bad input using a numeric column
air_check.count_string_levels("T")

Column 'T' is numeric.


In [21]:
# Test a bad input where the second column is numeric
air_check.count_string_levels("time_period", "RH")

Column 'RH' is numeric.


**Example using object created from pandas**

In [39]:
# One example using the object created from a pandas DataFrame
result_pd = air_check_pd.summarize_numeric_min_max("T")

# Check the output type
print(type(result_pd))

# Show the result
result_pd

<class 'pandas.core.frame.DataFrame'>


,T_min,T_max
0,-200.0,44.6


## Part II - NFL Analysis

In this section, I analyze weekly NFL quarterback data using two Spark APIs: pandas-on-Spark and Spark SQL DataFrames. I use the same filtering and summary steps in both approaches so I can compare the results.

In [7]:
# Set this before importing pandas-on-Spark
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Import pandas-on-Spark
import pyspark.pandas as ps

**Part IIA: pandas-on-Spark**

In [9]:
# Read with an explicit index column to avoid the default-index warning
nfl_ps = ps.read_csv("weekly_nfl_data.csv", index_col="player_id")

# Show the first 5 rows
nfl_ps.head(5)

,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
player_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


In [10]:
# Show all column names
list(nfl_ps.columns)

['player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

In [11]:
# Keep only QB regular-season rows from 2005 to 2023
qb_ps = nfl_ps[
    (nfl_ps["position"] == "QB") &
    (nfl_ps["season_type"] == "REG") &
    (nfl_ps["season"] >= 2005) &
    (nfl_ps["season"] <= 2023)
][[
    "player_display_name",
    "season",
    "week",
    "completions",
    "attempts",
    "passing_yards",
    "passing_tds",
    "interceptions"
]]

# Show first few filtered rows
qb_ps.head(5)

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
player_id,,,,,,,,
00-0000722,Tony Banks,2005,17,14,25,173.0,1,2.0
00-0000865,Charlie Batch,2005,9,9,16,65.0,0,1.0
00-0000865,Charlie Batch,2005,10,13,19,150.0,0,0.0
00-0000865,Charlie Batch,2005,16,1,1,31.0,1,0.0
00-0001335,Jeff Blake,2005,2,1,1,11.0,0,0.0


In [12]:
# Group by player and season and compute sums and means
qb_summary_ps = qb_ps.groupby(["player_display_name", "season"]).agg({
    "week": ["sum", "mean"],
    "completions": ["sum", "mean"],
    "attempts": ["sum", "mean"],
    "passing_yards": ["sum", "mean"],
    "passing_tds": ["sum", "mean"],
    "interceptions": ["sum", "mean"]
}).reset_index()

# Flatten multi-level column names
new_cols = []
for col_name in qb_summary_ps.columns:
    if isinstance(col_name, tuple):
        left, right = col_name
        if right == "":
            new_cols.append(left)
        else:
            new_cols.append(f"{left}_{right}")
    else:
        new_cols.append(col_name)

qb_summary_ps.columns = new_cols

# Create the required variables
qb_summary_ps["completion_percentage"] = (
    qb_summary_ps["completions_sum"] / qb_summary_ps["attempts_sum"]
)

qb_summary_ps["td_int_ratio"] = (
    qb_summary_ps["passing_tds_sum"] / qb_summary_ps["interceptions_sum"]
)

# Show the first few rows
qb_summary_ps.head(5)

,player_display_name,season,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,99,7.615385,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,144,9.000000,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
2,Matt Schaub,2006,60,12.000000,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,0.666667,0.500000
3,Vince Young,2006,143,9.533333,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,48,8.000000,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN


In [13]:
# Keep only player-season rows with at least 50 attempts
qb_summary_ps_50 = qb_summary_ps[qb_summary_ps["attempts_sum"] >= 50]

# Show first few rows
qb_summary_ps_50.head(5)

,player_display_name,season,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,99,7.615385,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,144,9.000000,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
3,Vince Young,2006,143,9.533333,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,48,8.000000,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN
6,Charlie Batch,2006,71,10.142857,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,inf


In [14]:
# Top 40 by completion percentage
qb_summary_ps_50.sort_values("completion_percentage", ascending=False).head(40)

,player_display_name,season,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,65,10.833333,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,0.754717,inf
1248,Colt McCoy,2021,62,8.857143,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,0.747475,3.000000
900,Matt Schaub,2019,52,10.400000,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,0.746269,3.000000
953,Drew Brees,2018,130,8.666667,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,119,10.818182,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1338,Mason Rudolph,2023,66,16.500000,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,0.743243,inf
1133,Taysom Hill,2020,147,9.187500,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,0.727273,2.000000
1007,Nick Foles,2018,51,10.200000,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,0.723077,1.750000
917,Drew Brees,2017,148,9.250000,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,146,9.733333,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000


In [15]:
# Top 40 by TD to INT ratio
qb_summary_ps_50.sort_values("td_int_ratio", ascending=False).head(40)

,player_display_name,season,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
6,Charlie Batch,2006,71,10.142857,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,inf
26,Matt Schaub,2005,65,9.285714,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,0.515625,inf
73,Todd Collins,2007,62,15.500000,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,0.638095,inf
455,Troy Smith,2007,62,15.500000,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,0.526316,inf
520,Jake Locker,2011,51,10.200000,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,0.515152,inf
775,Brian Hoyer,2016,27,4.500000,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,0.670000,inf
778,Nick Foles,2016,17,8.500000,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,0.654545,inf
812,Derek Anderson,2014,30,6.000000,65,13.000000,97,19.400000,701.0,140.200000,5,1.000000,0.0,0.000000,0.670103,inf
940,Jimmy Garoppolo,2016,49,8.166667,43,7.166667,63,10.500000,502.0,83.666667,4,0.666667,0.0,0.000000,0.682540,inf
984,Matt Moore,2019,54,9.000000,59,9.833333,91,15.166667,659.0,109.833333,4,0.666667,0.0,0.000000,0.648352,inf


**Part IIB: Spark SQL DataFrame API**

In [16]:
# Read the weekly NFL data using Spark SQL
nfl_spark = spark.read.load(
    "weekly_nfl_data.csv",
    format="csv",
    sep=",",
    inferSchema=True,
    header=True
)

# Show the first 5 rows
nfl_spark.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [17]:
# Show all column names
nfl_spark.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

In [18]:
# Keep only QB regular-season rows from 2005 to 2023
qb_spark = (
    nfl_spark
    .filter(
        (F.col("position") == "QB") &
        (F.col("season_type") == "REG") &
        (F.col("season") >= 2005) &
        (F.col("season") <= 2023)
    )
    .select(
        "player_display_name",
        "season",
        "week",
        "completions",
        "attempts",
        "passing_yards",
        "passing_tds",
        "interceptions"
    )
)

# Show first few filtered rows
qb_spark.show(5)

+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|          1|       1|         11.0|          0|          0.0|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
only showing top 5 rows


In [19]:
# Group by player and season and compute sums and means
qb_summary_spark = (
    qb_spark
    .groupBy("player_display_name", "season")
    .agg(
        F.sum("week").alias("week_sum"),
        F.mean("week").alias("week_mean"),
        F.sum("completions").alias("completions_sum"),
        F.mean("completions").alias("completions_mean"),
        F.sum("attempts").alias("attempts_sum"),
        F.mean("attempts").alias("attempts_mean"),
        F.sum("passing_yards").alias("passing_yards_sum"),
        F.mean("passing_yards").alias("passing_yards_mean"),
        F.sum("passing_tds").alias("passing_tds_sum"),
        F.mean("passing_tds").alias("passing_tds_mean"),
        F.sum("interceptions").alias("interceptions_sum"),
        F.mean("interceptions").alias("interceptions_mean")
    )
    .withColumn(
        "completion_percentage",
        F.col("completions_sum") / F.col("attempts_sum")
    )
    .withColumn(
        "td_int_ratio",
        F.col("passing_tds_sum") / F.col("interceptions_sum")
    )
)

# Show first few rows
qb_summary_spark.show(5)

+-------------------+------+--------+-----------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|player_display_name|season|week_sum|        week_mean|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum|interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+--------+-----------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|      99|7.615384615384615|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|1.3076

In [20]:
# Keep only player-season rows with at least 50 attempts
qb_summary_spark_50 = qb_summary_spark.filter(F.col("attempts_sum") >= 50)

In [21]:
# Top 40 by completion percentage
qb_summary_spark_50.orderBy(F.col("completion_percentage").desc()).show(40, truncate=False)

+-------------------+------+--------+------------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|player_display_name|season|week_sum|week_mean         |completions_sum|completions_mean  |attempts_sum|attempts_mean     |passing_yards_sum|passing_yards_mean|passing_tds_sum|passing_tds_mean   |interceptions_sum|interceptions_mean |completion_percentage|td_int_ratio      |
+-------------------+------+--------+------------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|C.J. Beathard      |2023  |65      |10.833333333333334|40             |6.666666666666667 |53          |8.833333333333334 |349.0            |58.166666666666664|1           

In [22]:
# Top 40 by TD to INT ratio
qb_summary_spark_50.orderBy(F.col("td_int_ratio").desc()).show(40, truncate=False)

+-------------------+------+--------+------------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+
|player_display_name|season|week_sum|week_mean         |completions_sum|completions_mean  |attempts_sum|attempts_mean     |passing_yards_sum|passing_yards_mean|passing_tds_sum|passing_tds_mean  |interceptions_sum|interceptions_mean |completion_percentage|td_int_ratio     |
+-------------------+------+--------+------------------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+
|Tom Brady          |2016  |134     |11.166666666666666|291            |24.25             |432         |36.0              |3554.0           |296.1666666666667 |28             |2.

**Comparison**

The completion percentage results should be very similar across the two APIs.

The touchdown-to-interception ratio can behave differently when `interceptions_sum` is 0:
- pandas-on-Spark may show `inf`
- Spark SQL may show `NULL`